## Evaluation Framework

In order to set up an evaluation, we need to consider what we are aiming to get. Down the line, the ESCO nodes found by our vector search will be used to build a prompt so that the Large Language Model can ask users to validate a number of skills and occupations found through vector search. For that reason, we don't want to focus so much on the precision (that is, reducing the amount of false positives), but rather on the recall (that is, reducing the amount of false negatives). 

Differently from the occupation experiments, we now have that multiple skills can correspond to a single sentence. For this reason, we want to understand both the average precision@k and the average recall@k, focusing on the latter which is more important for our application. The precision@k evaluates how many of the top k retrieved nodes are true positives, penalizing the use of larger sample sizes. The recall@k, instead, focuses on calculating how many of the total true positives in a given sample are retrieved in the top k classes. This actually penalizes lower values of k, as less nodes will be retrieved.

In [1]:
from typing import List

def precision_at_k(prediction: List[List[str]], true: List[List[str]], k:int):
    """Calculates the average precision at k considering
    for each prediction the number of correct retrieved nodes
    divided by the number of total retrieved nodes.

    Args:
        prediction (List[List[str]]): list of 
            predicted lists, each with the corresponding
            nodes.
        true (List[List[str]]): list of the multiple true nodes 
            for each sample in the dataset.

    Returns:
        float: average precision at k over all the test set.
    """
    assert len(prediction) == len(true)
    total_precision = 0
    for pred_list, true_val in zip(prediction, true):
        total_precision+=len(set(pred_list[:k]).intersection(set(true_val)))/k
    return total_precision/len(true)

def recall_at_k(prediction: List[List[str]], true: List[List[str]], k:int):
    """Calculates the average recall at k considering
    for each prediction the number of correct retrieved nodes
    divided by the number of total correct nodes.

    Args:
        prediction (List[List[str]]): list of 
            predicted lists, each with the corresponding
            nodes.
        true (List[List[str]]): list of the multiple true nodes 
            for each sample in the dataset.

    Returns:
        float: average recall at k over all the test set.
    """
    assert len(prediction) == len(true)
    total_recall = 0
    for pred_list, true_val in zip(prediction, true):
        total_recall+=len(set(pred_list[:k]).intersection(set(true_val)))/len(set(true_val))
    return total_recall/len(true)

## Evaluation goals

The objective of the evaluation is to choose which model and hyperparameters have the highest recall at k, while keeping an eye on the precision. For a given embedding model, the hyperparameters are the following:

1. **How to embed a node of the graph**: which combination of the fields guarantees the best performance when embedded.
2. **Score function**: which function should be used to retrieve the most similar nodes (cosine, l2 distance or scalar product).
3. **Using Maximal Marginal Relevance**: whether we should use MMR to get more diverse results.

We will use ChromaDB as a local vector store and get the ESCO data from a local csv file. We will restrict our evaluation to the gecko003 model, but this can be repeated with any other model.

In [12]:
# 1. Loading the test dataset for skills using the Huggingface library
from huggingface_hub import hf_hub_download
import pandas as pd
from tqdm import tqdm
import os 
from dotenv import load_dotenv
from vertexai.language_models import TextEmbeddingModel

tqdm.pandas()

load_dotenv("/Users/francescopreta/coding/compass/backend/.env")

HF_TOKEN = os.environ["HF_ACCESS_TOKEN"]
REPO_ID = "tabiya/esco_skills_test"
FILENAME = "data/processed_skill_test_set_with_id.parquet"

df_test = pd.read_parquet(
    hf_hub_download(repo_id=REPO_ID, filename=FILENAME, repo_type="dataset", token=HF_TOKEN)
)

SKILL_CSV_PATH = "/Users/francescopreta/coding/compass/backend/esco_search/_scripts/skills.csv"
df_database = pd.read_csv(SKILL_CSV_PATH)

model = TextEmbeddingModel.from_pretrained("textembedding-gecko@003")

In what follows, we will pre-compute the strings and the corresponding embeddings using the Gecko model. We will use manual batching to speed up the process, as the vertex API doesn't support batching and fails if the list length is larger than 250 elements or the sum of tokens is higher than 20.000.

In [3]:
def embed_strings_in_batch(list_of_queries: List[str], model: TextEmbeddingModel, batch_size: int = 100) -> List[List[float]]:
    """Uses a TextEmbeddingModel to embed a list of queries in batches.

    Args:
        list_of_queries (List[str]): list of queries to be embedded in batches.
        model (TextEmbeddingModel): embedding model.
        batch_size (int, optional): size of each batch which should be less than or equal to 250.
            Defaults to 100.

    Returns:
        List[List[float]]: List of embeddings corresponding to the queries.
    """
    assert batch_size<=250
    embedding_results = []
    num_samples = len(list_of_queries)
    for i in range(int(num_samples/batch_size)+1):
        batch = list_of_queries[i*batch_size:(i+1)*batch_size]
        embedding_results += model.get_embeddings(batch)
    assert len(embedding_results) == len(list_of_queries)
    return [embedding_result.values for embedding_result in embedding_results]

We define the functions generating the string that can be embedded from each document and save their computation in a dictionary. Then we save the corresponding embeddings in another dictionary, as well as the embeddings of the test set, that will be used for query retrieval.

In [5]:
# Functions defining strings to embed
def description(df):
    return df["DESCRIPTION"]

def preferred_label(df):
    return df["PREFERREDLABEL"]

def all_skills(df):
    return f"""Skill Names: {df['PREFERREDLABEL']}
{df['ALTLABELS']}

Skill Description: {df['DESCRIPTION']}"""

def label_and_description(df):
    return f"{df['PREFERREDLABEL']}\n{df['DESCRIPTION']}"

function_to_method ={"DESCRIPTION": description, "PREFERREDLABEL":preferred_label, "ALL_SKILLS":all_skills, "LABEL_AND_DESCRIPTION": label_and_description}

In [6]:
# Database embedding precomputation
method_to_embeddings = {}
method_to_documents = {}
for method in function_to_method:
    method_to_documents[method] = list(df_database.progress_apply(function_to_method[method], axis=1))
    method_to_embeddings[method] = embed_strings_in_batch(method_to_documents[method], model)

100%|██████████| 13896/13896 [00:00<00:00, 347114.81it/s]


In [13]:
# Test set preprocessing
df_test = df_test.groupby(['synthetic_query', 'sentence']).agg(list).reset_index()
test_embeddings = embed_strings_in_batch(list(df_test["synthetic_query"]), model)

In [15]:
# Functions "maximal_marginal_relevance" and "cosine_similarity"
# are duplicated respectively from modules:
#    - "libs/community/langchain_community/vectorstores/utils.py"
#    - "libs/community/langchain_community/utils/math.py"
from typing import List, Union

import numpy as np

Matrix = Union[List[List[float]], List[np.ndarray], np.ndarray]


def cosine_similarity(X: Matrix, Y: Matrix) -> np.ndarray:
    """Row-wise cosine similarity between two equal-width matrices."""
    if len(X) == 0 or len(Y) == 0:
        return np.array([])

    X = np.array(X)
    Y = np.array(Y)
    if X.shape[1] != Y.shape[1]:
        raise ValueError(
            f"Number of columns in X and Y must be the same. X has shape {X.shape} "
            f"and Y has shape {Y.shape}."
        )
    X_norm = np.linalg.norm(X, axis=1)
    Y_norm = np.linalg.norm(Y, axis=1)
    # Ignore divide by zero errors run time warnings as those are handled below.
    with np.errstate(divide="ignore", invalid="ignore"):
        similarity = np.dot(X, Y.T) / np.outer(X_norm, Y_norm)
    similarity[np.isnan(similarity) | np.isinf(similarity)] = 0.0
    return similarity



def maximal_marginal_relevance(
    query_embedding: np.ndarray,
    embedding_list: list,
    lambda_mult: float = 0.5,
    k: int = 10,
) -> List[int]:
    """Calculate maximal marginal relevance."""
    if min(k, len(embedding_list)) <= 0:
        return []
    if query_embedding.ndim == 1:
        query_embedding = np.expand_dims(query_embedding, axis=0)
    similarity_to_query = cosine_similarity(query_embedding, embedding_list)[0]
    most_similar = int(np.argmax(similarity_to_query))
    idxs = [most_similar]
    selected = np.array([embedding_list[most_similar]])
    while len(idxs) < min(k, len(embedding_list)):
        best_score = -np.inf
        idx_to_add = -1
        similarity_to_selected = cosine_similarity(embedding_list, selected)
        for i, query_score in enumerate(similarity_to_query):
            if i in idxs:
                continue
            redundant_score = max(similarity_to_selected[i])
            equation_score = (
                lambda_mult * query_score - (1 - lambda_mult) * redundant_score
            )
            if equation_score > best_score:
                best_score = equation_score
                idx_to_add = i
        idxs.append(idx_to_add)
        selected = np.append(selected, [embedding_list[idx_to_add]], axis=0)
    return idxs


We create multiple chromadb collections to store our data in memory with different embeddings depending on the method used and on the function used for querying. On these, we save the Occupation ESCO database with all their metadatas.

In [17]:
import chromadb

client = chromadb.Client()
embedding_methods = ["DESCRIPTION", "PREFERREDLABEL", "ALL_SKILLS", "LABEL_AND_DESCRIPTION"]
score_function = ["cosine", "l2", "ip"]
for method in embedding_methods:
    for score in score_function:
        collection_name = f'{method}_{score}'
        collection_metadata = {"hnsw:space":score}
        collection = client.create_collection(name=collection_name, metadata=collection_metadata)
        collection.add(
            documents = method_to_documents[method],
            metadatas = df_database.to_dict('records'),
            embeddings = method_to_embeddings[method],
            ids = [f"id_{i}" for i in range(len(df_database))]
        )

Finally, we run the evaluation as follows:

1. We choose a score function and a method and load the corresponding collection.
2. For each element in the test set, we find the top 100 documents in the collection ordered by scoring rank.
3. We filter those documents by maximal marginal relevance to find the top 10 documents with this function.
4. We evaluate the precision and recall on the top k for k=1,3,5,10 for the original retrieved documents.
5. We evaluate the precision and recall on the top k for k=1,3,5,10 for the documents filtered by maximal marginal relevance.
6. We save the results in a dataframe to be analyzed for later use.

In [18]:
df_database.columns

Index(['ORIGINURI', 'ID', 'UUIDHISTORY', 'SKILLTYPE', 'REUSELEVEL',
       'PREFERREDLABEL', 'ALTLABELS', 'DESCRIPTION', 'DEFINITION',
       'SCOPENOTE'],
      dtype='object')

In [19]:
eval_data = []
for method in embedding_methods:
    for score in score_function:
        collection_name = f'{method}_{score}'
        # Fetch collection
        collection = client.get_collection(name=collection_name)
        # Initialize lists to save results
        vector_search_results = []
        mmr_vector_search_results = []
        for test_embedding in test_embeddings:
            # Find the top 100 documents and save them in vector_search_results
            documents_from_search = collection.query(query_embeddings=test_embedding, n_results=100, include=["metadatas", "embeddings"])
            vector_search_results.append([elem["UUIDHISTORY"] for elem in documents_from_search["metadatas"][0]])
            # Find the top 10 documents according to MMR and save them in mmr_vector_search_results
            result_embeddings = [elem for elem in documents_from_search["embeddings"][0]]
            mmr_ids = maximal_marginal_relevance(np.array(test_embedding), result_embeddings)
            mmr_vector_search_results.append([elem["UUIDHISTORY"] for index, elem in enumerate(documents_from_search["metadatas"][0]) if index in mmr_ids])
        # Evaluate accuracy at k for k=1, 3, 5, 10
        for k in [1, 3, 5, 10]:
            prec_at_k = precision_at_k(vector_search_results, list(df_test["UUID"]), k)
            rec_at_k = recall_at_k(vector_search_results, list(df_test["UUID"]), k)
            eval_data.append({"method":method, "score function":score, "MMR": False, "k":k, "precision": prec_at_k, "recall": rec_at_k})
            print(f"Method: {method}, Score function: {score}, MMR: False\n Precision at {k}: {round(prec_at_k,4)}, Recall at {k}: {round(rec_at_k,4)}")
            prec_at_k = precision_at_k(mmr_vector_search_results, list(df_test["UUID"]), k)
            rec_at_k = recall_at_k(mmr_vector_search_results, list(df_test["UUID"]), k)
            eval_data.append({"method":method, "score function":score, "MMR": True, "k":k, "precision": prec_at_k, "recall": rec_at_k})
            print(f"Method: {method}, Score function: {score}, MMR: True\n Precision at {k}: {round(prec_at_k,4)}, Recall at {k}: {round(rec_at_k,4)}")
# Save the results in a dataframe
eval_df = pd.DataFrame(eval_data)

Method: ALL_SKILLS, Score function: cosine, MMR: False
 Precision at 1: 0.2841, Recall at 1: 0.1977


KeyError: 'esco_code'

In [13]:
# Save the evaluation results locally
OUTPUT_EVAL_PATH = "/Users/francescopreta/coding/compass/backend/esco_search/_scripts/skills_eval_results.csv"
eval_df.to_csv(OUTPUT_EVAL_PATH)

The evaluation results are as follows:

| method            | score function | MMR  | k  | accuracy           |
|-------------------|----------------|------|----|--------------------|
| PREFERREDLABEL    | cosine         | False| 10 | 0.734545454545455  |
| PREFERREDLABEL    | l2             | False| 10 | 0.734545454545455  |
| PREFERREDLABEL    | ip             | False| 10 | 0.734545454545455  |
| ALL_OCCUPATIONS   | cosine         | False| 10 | 0.727272727272727  |
| ALL_OCCUPATIONS   | l2             | False| 10 | 0.727272727272727  |
| ALL_OCCUPATIONS   | ip             | False| 10 | 0.727272727272727  |
| LABEL_AND_DESCRIPTION| cosine      | False| 10 | 0.709090909090909  |
| LABEL_AND_DESCRIPTION| l2          | False| 10 | 0.709090909090909  |
| LABEL_AND_DESCRIPTION| ip          | False| 10 | 0.709090909090909  |
| DESCRIPTION       | cosine         | False| 10 | 0.703636363636364  |
| DESCRIPTION       | l2             | False| 10 | 0.703636363636364  |
| DESCRIPTION       | ip             | False| 10 | 0.703636363636364  |
| PREFERREDLABEL    | cosine         | False| 5  | 0.652727272727273  |
| PREFERREDLABEL    | l2             | False| 5  | 0.652727272727273  |
| PREFERREDLABEL    | ip             | False| 5  | 0.652727272727273  |
| ALL_OCCUPATIONS   | cosine         | False| 5  | 0.650909090909091  |
| ALL_OCCUPATIONS   | l2             | False| 5  | 0.650909090909091  |
| ALL_OCCUPATIONS   | ip             | False| 5  | 0.650909090909091  |
| DESCRIPTION       | cosine         | False| 5  | 0.610909090909091  |
| DESCRIPTION       | l2             | False| 5  | 0.610909090909091  |
| DESCRIPTION       | ip             | False| 5  | 0.610909090909091  |
| LABEL_AND_DESCRIPTION| cosine      | False| 5  | 0.605454545454546  |
| LABEL_AND_DESCRIPTION| l2          | False| 5  | 0.605454545454546  |
| LABEL_AND_DESCRIPTION| ip          | False| 5  | 0.605454545454546  |
| PREFERREDLABEL    | cosine         | False| 3  | 0.572727272727273  |
| PREFERREDLABEL    | l2             | False| 3  | 0.572727272727273  |
| PREFERREDLABEL    | ip             | False| 3  | 0.572727272727273  |
| ALL_OCCUPATIONS   | cosine         | False| 3  | 0.572727272727273  |
| ALL_OCCUPATIONS   | l2             | False| 3  | 0.572727272727273  |
| ALL_OCCUPATIONS   | ip             | False| 3  | 0.572727272727273  |
| LABEL_AND_DESCRIPTION| cosine      | False| 3  | 0.54               |
| LABEL_AND_DESCRIPTION| l2          | False| 3  | 0.54               |
| LABEL_AND_DESCRIPTION| ip          | False| 3  | 0.54               |
| DESCRIPTION       | cosine         | False| 3  | 0.536363636363636  |
| DESCRIPTION       | l2             | False| 3  | 0.536363636363636  |
| DESCRIPTION       | ip             | False| 3  | 0.536363636363636  |
| PREFERREDLABEL    | l2             | True | 10 | 0.443636363636364  |
| PREFERREDLABEL    | cosine         | True | 10 | 0.441818181818182  |
| PREFERREDLABEL    | ip             | True | 10 | 0.441818181818182  |
| PREFERREDLABEL    | cosine         | True | 5  | 0.44               |
| PREFERREDLABEL    | l2             | True | 5  | 0.44               |
| PREFERREDLABEL    | ip             | True | 5  | 0.44               |
| ALL_OCCUPATIONS   | cosine         | True | 10 | 0.427272727272727  |
| ALL_OCCUPATIONS   | l2             | True | 10 | 0.427272727272727  |
| ALL_OCCUPATIONS   | ip             | True | 10 | 0.427272727272727  |
| LABEL_AND_DESCRIPTION| cosine      | True | 10 | 0.425454545454546  |
| LABEL_AND_DESCRIPTION| l2          | True | 10 | 0.425454545454546  |
| LABEL_AND_DESCRIPTION| ip          | True | 10 | 0.425454545454546  |
| LABEL_AND_DESCRIPTION| cosine      | True | 5  | 0.425454545454546  |
| LABEL_AND_DESCRIPTION| l2          | True | 5  | 0.425454545454546  |
| LABEL_AND_DESCRIPTION| ip          | True | 5  | 0.425454545454546  |
| PREFERREDLABEL    | cosine         | True | 3  | 0.425454545454546  |
| PREFERREDLABEL    | l2             | True | 3  | 0.425454545454546  |
| PREFERREDLABEL    | ip             | True | 3  | 0.425454545454546  |
| ALL_OCCUPATIONS   | cosine         | True | 5  | 0.418181818181818  |
| ALL_OCCUPATIONS   | l2             | True | 5  | 0.418181818181818  |
| ALL_OCCUPATIONS   | ip             | True | 5  | 0.418181818181818  |
| LABEL_AND_DESCRIPTION| cosine      | True | 3  | 0.416363636363636  |
| LABEL_AND_DESCRIPTION| l2          | True | 3  | 0.416363636363636  |
| LABEL_AND_DESCRIPTION| ip          | True | 3  | 0.416363636363636  |
| ALL_OCCUPATIONS   | cosine         | True | 3  | 0.410909090909091  |
| ALL_OCCUPATIONS   | l2             | True | 3  | 0.410909090909091  |
| ALL_OCCUPATIONS   | ip             | True | 3  | 0.410909090909091  |
| DESCRIPTION       | cosine         | True | 10 | 0.394545454545455  |
| DESCRIPTION       | l2             | True | 10 | 0.394545454545455  |
| DESCRIPTION       | ip             | True | 10 | 0.394545454545455  |
| DESCRIPTION       | cosine         | True | 5  | 0.390909090909091  |
| DESCRIPTION       | l2             | True | 5  | 0.390909090909091  |
| DESCRIPTION       | ip             | True | 5  | 0.390909090909091  |
| DESCRIPTION       | cosine         | True | 3  | 0.38               |
| DESCRIPTION       | l2             | True | 3  | 0.38               |
| DESCRIPTION       | ip             | True | 3  | 0.38               |
| PREFERREDLABEL    | cosine         | False| 1  | 0.374545454545455  |
| PREFERREDLABEL    | cosine         | True | 1  | 0.374545454545455  |
| PREFERREDLABEL    | l2             | False| 1  | 0.374545454545455  |
| PREFERREDLABEL    | l2             | True | 1  | 0.374545454545455  |
| PREFERREDLABEL    | ip             | False| 1  | 0.374545454545455  |
| PREFERREDLABEL    | ip             | True | 1  | 0.374545454545455  |
| LABEL_AND_DESCRIPTION| cosine      | False| 1  | 0.367272727272727  |
| LABEL_AND_DESCRIPTION| cosine      | True | 1  | 0.367272727272727  |
| LABEL_AND_DESCRIPTION| l2          | False| 1  | 0.367272727272727  |
| LABEL_AND_DESCRIPTION| l2          | True | 1  | 0.367272727272727  |
| LABEL_AND_DESCRIPTION| ip          | False| 1  | 0.367272727272727  |
| LABEL_AND_DESCRIPTION| ip          | True | 1  | 0.367272727272727  |
| ALL_OCCUPATIONS   | cosine         | False| 1  | 0.341818181818182  |
| ALL_OCCUPATIONS   | cosine         | True | 1  | 0.341818181818182  |
| ALL_OCCUPATIONS   | l2             | False| 1  | 0.341818181818182  |
| ALL_OCCUPATIONS   | l2             | True | 1  | 0.341818181818182  |
| ALL_OCCUPATIONS   | ip             | False| 1  | 0.341818181818182  |
| ALL_OCCUPATIONS   | ip             | True | 1  | 0.341818181818182  |
| DESCRIPTION       | cosine         | False| 1  | 0.336363636363636  |
| DESCRIPTION       | cosine         | True | 1  | 0.336363636363636  |
| DESCRIPTION       | l2             | False| 1  | 0.336363636363636  |
| DESCRIPTION       | l2             | True | 1  | 0.336363636363636  |
| DESCRIPTION       | ip             | False| 1  | 0.336363636363636  |
| DESCRIPTION       | ip             | True | 1  | 0.336363636363636  |


The result of the evaluation are as follows:

1. The results obtained without MMR are definitely better than the results obtained without MMR. This happens because the correct code is most of the times within the first k elements and still very similar to the first one. MMR excludes many good high ranking results that could be retrieved otherwise because they are too similar to the first result.
2. Different retrieval functions return the same results. This probably happens because the Gecko embeddings are normalized so that l2, cosine similarity and dot product all determine the same ranking. We will choose the default version.
3. The best embedding methods are either PREFERREDLABEL or ALL_OCCUPATIONS judging from the results from 5 and 10. We are choosing PREFERREDLABEL given that it has a small margin over ALL_OCCUPATIONS and it's easier to implement as its value is already present in the dataset.

We can now write a function to embed the test and database dataframes and obtain the evaluation results.

In [ ]:
# Best results are given without MMR and using only the preferred label for inputs
def evaluate(model: TextEmbeddingModel, database: pd.DataFrame, test_set: pd.DataFrame, k: int) -> float:
    """Returns the accuracy at k for the vector search through the embedding generated 
    by a given model on a provided dataset. The model must be a TextEmbeddingModel,
    the database must be a Dataframe containing the columns 'PREFERREDLABEL' and 'CODE',
    while the test set must be a Dataframe containing the columns 'synthetic_query' and
    'esco_code'.

    Args:
        model (TextEmbeddingModel): model to be evaluated.
        database (pd.DataFrame): database from which the data should be retrieved.
        test_set (pd.DataFrame): test set for the evaluation.
        k (int): k for the accuracy at k to be evaluated.

    Returns:
        float: accuracy at k
    """
    assert k<=100
    # Calculate embeddings for the database
    db_embeddings = embed_strings_in_batch(list(database['PREFERREDLABEL']), model)
    # Calculate embeddings for the test set
    test_embeddings = embed_strings_in_batch(list(test_set['synthetic_query']), model)
    # Load a collection from chromadb and saves the data
    client = chromadb.Client()
    collection = client.create_collection(name='eval')
    collection.add(
        documents = list(database['PREFERREDLABEL']),
        metadatas = [{"CODE": row["CODE"]} for _, row in database.iterrows()],
        embeddings = db_embeddings,
        ids = [f"id_{i}" for i in range(len(database))]
    )
    # Save the data retrieved from the test set
    vector_search_results = []
    for test_embedding in test_embeddings:
        # Find the top 100 documents and save them in vector_search_results
        documents_from_search = collection.query(query_embeddings=test_embedding, n_results=100, include=["metadatas", "embeddings"])
        vector_search_results.append([elem["CODE"] for elem in documents_from_search["metadatas"][0]])
    # Evaluate accuracy at k 
    return accuracy_at_k(vector_search_results, list(test_set["esco_code"]), k)
